In [23]:
using JLD2
using Flux
using ONNXNaiveNASflux
using Random
using Cersyve

In [36]:
task = Unicycle
task_name = "unicycle"

value_hidden_sizes = [32, 32]
dynamics_hidden_sizes = [32, 32]
constraint_hidden_sizes = [16]
data_path = joinpath(@__DIR__, "../data/$(task_name)_data.jld2")
model_dir = joinpath(@__DIR__, "../model/$task_name/")
onnx_dir = joinpath(@__DIR__, "../onnx/")

V_model = Cersyve.create_mlp(task.x_dim, 1, value_hidden_sizes)
Flux.loadmodel!(V_model, JLD2.load(joinpath(model_dir, "V_pretrain.jld2"), "state"))
ONNXNaiveNASflux.save(joinpath(onnx_dir, "$(task_name)_V_pretrain.onnx"), V_model, (task.x_dim, 1))
Flux.loadmodel!(V_model, JLD2.load(joinpath(model_dir, "V_finetune.jld2"), "state"))
ONNXNaiveNASflux.save(joinpath(onnx_dir, "$(task_name)_V_finetune.onnx"), V_model, (task.x_dim, 1))

data = JLD2.load(data_path)["data"]
f_model = Cersyve.create_mlp(task.x_dim + task.u_dim, task.x_dim, dynamics_hidden_sizes)
Flux.loadmodel!(f_model, JLD2.load(joinpath(model_dir, "f.jld2"), "state"))
f_pi_model = Cersyve.create_closed_loop_dynamics_model(
    f_model, task.pi_model, data, task.x_low, task.x_high, task.u_dim)
ONNXNaiveNASflux.save(joinpath(onnx_dir, "$(task_name)_f.onnx"), f_pi_model, (task.x_dim, 1))

h_model = Cersyve.create_mlp(task.x_dim, 1, constraint_hidden_sizes)
Flux.loadmodel!(h_model, JLD2.load(joinpath(model_dir, "h.jld2"), "state"))
ONNXNaiveNASflux.save(joinpath(onnx_dir, "$(task_name)_h.onnx"), h_model, (task.x_dim, 1))

760